# Advanced Python Tutorial: Designing `__format__` Through Problems

**A new, step-by-step problem set with complete solutions.** Each problem is broken into a specification, design decisions, small implementation stages, demonstrations, tests, and a recap. The examples deliberately differ from a basic `Person`/date-formatting demonstration.

**Audience:** Python programmers comfortable with classes, properties, exceptions, and f-strings. **Runtime:** Python 3.10+; standard library only. **How to use:** Run the cells from top to bottom. Read the problem statement before revealing the solution stages, then try writing each stage yourself. All examples use deterministic values rather than the current clock or locale-dependent formatting.

### Learning goals

- Understand how `format(value, spec)` and f-strings find `type(value).__format__`.
- Invent small, documented, strictly validated format-specification languages.
- Delegate standard numeric, string, and datetime formatting rather than reimplementing it.
- Address precision, signs, negative zero, time zones, binary units, and round-trip safety.
- Compose independently formattable classes into reports without confusing display strings with machine data.

> **Design rule:** Every implementation of `__format__(self, format_spec)` must return a `str`. For invalid *custom* specifications, raise a useful `ValueError`. Keep the empty specification meaningful and documented.

## Warm-up — what actually happens during formatting?

Three operations look similar but are not interchangeable:

```python
str(obj)                 # calls the object's string representation
repr(obj)                # calls the object's representation
format(obj, 'some_spec') # calls the type's __format__ method
```

An f-string such as `f'{obj:some_spec}'` also uses `__format__`. An explicit conversion, such as `!s` or `!r`, first produces a **string**, so any subsequent width/alignment specification is handled by **`str.__format__`**, not by your object's `__format__`.

A class that does not override `object.__format__` supports the empty specification (using `str(obj)`), but typically rejects a nonempty specification with `TypeError`. Let's establish this baseline before solving the problems.

In [1]:
class PlainObject:
    def __str__(self):
        return "plain object"

plain = PlainObject()
assert format(plain) == "plain object"
assert f"{plain}" == "plain object"

try:
    format(plain, ">20")
except TypeError as exc:
    print("Expected error from object.__format__:", exc)
else:
    raise AssertionError("A nonempty spec unexpectedly succeeded")

Expected error from object.__format__: unsupported format string passed to PlainObject.__format__


The rest of the notebook uses small assertion-based tests. They are intentionally plain Python, so the notebook remains usable without installing `pytest`.

In [2]:
def expect_error(exception_type, callback, containing=None):
    """Verify that a zero-argument callback raises the intended exception."""
    try:
        callback()
    except exception_type as exc:
        if containing is not None:
            assert containing in str(exc), (str(exc), containing)
        return str(exc)
    else:
        raise AssertionError(f"Expected {exception_type.__name__}")

print("Test helper ready")

Test helper ready


---
# Problem 1 — Trace the formatting protocol precisely

### Problem statement

Create a `ProtocolProbe` that records whether Python called `__str__`, `__repr__`, or `__format__`. It must delegate ordinary alignment to its underlying name. Explore four expressions:

1. `str(probe)`
2. `format(probe, '>14')`
3. `f'{probe!s:>14}'`
4. `f'{probe!r:>14}'`

**Challenge:** Demonstrate that assigning a `__format__` attribute directly to an *instance* does not override the special-method lookup used by `format()`.

Before running the solution, predict which method each expression reaches and where alignment is performed.

### Step 1 — instrument only the methods we need

This problem intentionally records method calls. In application code, formatting is usually best kept free of observable side effects; the event list is a teaching instrument, not a production pattern.

In [3]:
class ProtocolProbe:
    def __init__(self, name):
        self.name = name
        self.events = []

    def __str__(self):
        self.events.append("__str__")
        return f"STR({self.name})"

    def __repr__(self):
        self.events.append("__repr__")
        return f"ProtocolProbe({self.name!r})"

    def __format__(self, format_spec):
        self.events.append(("__format__", format_spec))
        # Reuse Python's already-correct string alignment implementation.
        return format(self.name, format_spec)

probe = ProtocolProbe("Ada")
print("Probe constructed; no formatting has happened yet.")
assert probe.events == []

Probe constructed; no formatting has happened yet.


### Step 2 — trace direct calls and the plain f-string

`str` and `repr` choose their own methods. Without `!s` or `!r`, an f-string calls `__format__` even for an **empty** specification.

In [4]:
probe.events.clear()
assert str(probe) == "STR(Ada)"
assert probe.events == ["__str__"]

probe.events.clear()
assert repr(probe) == "ProtocolProbe('Ada')"
assert probe.events == ["__repr__"]

probe.events.clear()
assert f"{probe}" == "Ada"
assert probe.events == [("__format__", "")]
print("str, repr, and empty f-string dispatch verified")

str, repr, and empty f-string dispatch verified


### Step 3 — compare conversions with the custom formatting path

The width in `>14` belongs to whichever object exists **after** any conversion. Thus `!s` and `!r` prevent `ProtocolProbe.__format__` from seeing the specifier.

In [5]:
probe.events.clear()
assert format(probe, ">14") == " " * 11 + "Ada"
assert probe.events == [("__format__", ">14")]

probe.events.clear()
assert f"{probe!s:>14}" == format("STR(Ada)", ">14")
assert probe.events == ["__str__"]

probe.events.clear()
assert f"{probe!r:>14}" == format("ProtocolProbe('Ada')", ">14")
assert probe.events == ["__repr__"]
print("Conversions and alignment verified")

Conversions and alignment verified


### Step 4 — special methods are looked up on the *type*

An ordinary `probe.__format__` attribute lookup can find an instance attribute. Python's built-in formatting protocol instead looks up the special method on its type. This distinction matters when mocking, subclassing, and debugging data models.

In [6]:
probe.__format__ = lambda spec: "INSTANCE OVERRIDE"
assert probe.__format__("whatever") == "INSTANCE OVERRIDE"  # ordinary lookup
probe.events.clear()
assert format(probe, ">5") == "  Ada"  # special-method lookup
assert probe.events == [("__format__", ">5")]
del probe.__format__
print("Instance-level override does not change format(probe, spec)")

Instance-level override does not change format(probe, spec)


### Solution recap

`format(obj, spec)` follows the type's `__format__`; `str(obj)` and `repr(obj)` are separate protocols. `f'{obj!r:spec}'` means "take `repr` first, then format the resulting string." Do not implement `__format__` by recursively formatting `self` or by calling `f'{self}'` inside it: that can recurse indefinitely.

---
# Problem 2 — build a small formatting grammar for exact fractions

### Problem statement

Implement a `FractionView` with three displays:

| Specifier | Meaning | Example for `-7/3` |
|---|---|---|
| empty | simplified numerator/denominator | `-7/3` |
| `mixed` | signed mixed number | `-2 1/3` |
| `pct\|N` | percentage to `N` decimal places | `-233.33%` with `pct\|2` |

Here `N` must be one of `0`, `1`, `2`, `3`, or `4`. Invalid specs must fail clearly. Use `fractions.Fraction` to normalize the rational number, and **`Decimal`**, not a binary float, to calculate the percentage display.

**Try first:** design the parsing branches before writing the numeric formatting code.

### Step 1 — store the mathematical value independently of presentation

The `Fraction` type reduces numerators and denominators and normalizes the denominator's sign. The wrapper adds only presentation behavior; it does not reimplement rational arithmetic.

In [7]:
from fractions import Fraction
from decimal import Decimal, localcontext

class FractionView:
    def __init__(self, numerator, denominator=1):
        self.value = Fraction(numerator, denominator)

    def __repr__(self):
        return (f"FractionView({self.value.numerator}, "
                f"{self.value.denominator})")

sample_fraction = FractionView(-14, 6)
assert sample_fraction.value == Fraction(-7, 3)
print(repr(sample_fraction))

FractionView(-7, 3)


### Step 2 — solve mixed-number rendering as a pure helper

Separate *parsing* the spec from *rendering* the value. For a negative fraction, divide the absolute numerator and restore a single leading minus sign. A proper fraction should not be displayed with a pointless `0` whole-number part.

In [8]:
def mixed_fraction_text(value):
    sign = "-" if value < 0 else ""
    whole, remainder = divmod(abs(value.numerator), value.denominator)
    if remainder == 0:
        return f"{sign}{whole}"
    if whole == 0:
        return f"{sign}{remainder}/{value.denominator}"
    return f"{sign}{whole} {remainder}/{value.denominator}"

assert mixed_fraction_text(Fraction(-7, 3)) == "-2 1/3"
assert mixed_fraction_text(Fraction(1, 3)) == "1/3"
assert mixed_fraction_text(Fraction(6, 3)) == "2"

### Step 3 — add the full `__format__` implementation

An exact rational still needs rounding when shown as a finite decimal percentage. Use local `Decimal` precision for the intermediate division and delegate the final fixed-point display to `Decimal.__format__`. This example uses the decimal context's default rounding mode (normally `ROUND_HALF_EVEN`).

In [9]:
def fraction_view_format(self, format_spec):
    value = self.value
    if format_spec == "":
        return f"{value.numerator}/{value.denominator}"
    if format_spec == "mixed":
        return mixed_fraction_text(value)
    if format_spec.startswith("pct|"):
        precision_text = format_spec.removeprefix("pct|")
        if precision_text not in {"0", "1", "2", "3", "4"}:
            raise ValueError("pct spec must be pct|N, where N is 0..4")
        precision = int(precision_text)
        with localcontext() as context:
            context.prec = 50
            percent = (Decimal(value.numerator) * 100
                       / Decimal(value.denominator))
            rendered = format(percent, f".{precision}f")
        return f"{rendered}%"
    raise ValueError(f"Unknown FractionView format: {format_spec!r}")

FractionView.__format__ = fraction_view_format

For pedagogical clarity, the implementation above was attached after the class definition so that each stage is visible. In a normal module, place the same `def __format__(...)` **inside** the class. The method is assigned on the **class**, unlike the ineffective instance assignment in Problem 1.

### Step 4 — demonstrate every branch, including invalid input

In [10]:
negative = FractionView(-7, 3)
assert format(negative) == "-7/3"
assert format(negative, "mixed") == "-2 1/3"
assert format(negative, "pct|2") == "-233.33%"
assert f"{FractionView(1, 8):pct|3}" == "12.500%"
assert format(FractionView(0, 5), "mixed") == "0"

for invalid in ("pct", "pct|", "pct|9", "mixed|2", "PCT|2"):
    print(invalid, "->", expect_error(ValueError,
          lambda invalid=invalid: format(negative, invalid)))

pct -> Unknown FractionView format: 'pct'
pct| -> pct spec must be pct|N, where N is 0..4
pct|9 -> pct spec must be pct|N, where N is 0..4
mixed|2 -> Unknown FractionView format: 'mixed|2'
PCT|2 -> Unknown FractionView format: 'PCT|2'


### Solution recap and extension

A tiny documented grammar is easier to maintain than guessing what arbitrary text means. The empty spec preserves an exact fraction, while `pct|N` explicitly opts into approximate decimal presentation. An extension exercise is to implement `decimal|N`, but keep the stored `Fraction` exact.

---
# Problem 3 — format money without binary-float surprises

### Problem statement

Design an immutable `Money` class with a `Decimal` amount and an ISO-style currency code. Implement four specs:

- empty or `code`: `USD -1,234.57`
- `symbol`: `-$1,234.57`
- `accounting`: `(USD 1,234.57)`
- `symbol-accounting`: `($1,234.57)`

This exercise deliberately supports **USD, EUR, and GBP only**; it is a constrained display example, not a general internationalization or currency-conversion library. Use two decimal places and `ROUND_HALF_EVEN`. Reject float inputs rather than silently converting approximate binary values into exact money.

**Edge cases:** negative zero after rounding, unsupported currencies, and invalid format specs.

### Step 1 — validate the model at construction

`Decimal('0.1')` is the exact decimal value users usually intend; `Decimal(0.1)` captures the existing float approximation. Requiring a `Decimal` prevents this common ambiguity from entering the model.

In [11]:
from dataclasses import dataclass
from decimal import ROUND_HALF_EVEN

CURRENCY_SYMBOLS = {"USD": "$", "EUR": "€", "GBP": "£"}

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

    def __post_init__(self):
        if not isinstance(self.amount, Decimal):
            raise TypeError("amount must be Decimal; construct it from a string")
        if not self.amount.is_finite():
            raise ValueError("amount must be finite")
        if self.currency not in CURRENCY_SYMBOLS:
            raise ValueError("supported currencies: USD, EUR, GBP")

usd = Money(Decimal("-1234.567"), "USD")
assert usd.amount == Decimal("-1234.567")
print("Stored value:", repr(usd))

Stored value: Money(amount=Decimal('-1234.567'), currency='USD')


### Step 2 — round once and normalize the sign

The formatting layer rounds the *display*, not the stored amount. Determine negativity **after** rounding: `-0.004` displays as `0.00`, not `-0.00`. Using `abs()` before formatting also makes accounting parentheses straightforward.

In [12]:
def render_money_parts(money):
    rounded = money.amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN)
    negative = rounded < 0  # -0.00 is not negative in this comparison.
    magnitude = format(abs(rounded), ",.2f")
    return negative, magnitude

assert render_money_parts(usd) == (True, "1,234.57")
assert render_money_parts(Money(Decimal("-0.004"), "USD")) == (False, "0.00")

### Step 3 — implement a closed set of presentation styles

Rather than trying to interpret every string mini-language option, accept only the documented modes. The standard `Decimal` formatter still does the work of grouping and decimal places.

In [13]:
def money_format(self, format_spec):
    style = format_spec or "code"
    allowed = {"code", "symbol", "accounting", "symbol-accounting"}
    if style not in allowed:
        raise ValueError(f"Money format must be one of {sorted(allowed)}")

    negative, magnitude = render_money_parts(self)
    symbol_style = style in {"symbol", "symbol-accounting"}
    accounting = style in {"accounting", "symbol-accounting"}
    base = (f"{CURRENCY_SYMBOLS[self.currency]}{magnitude}" if symbol_style
            else f"{self.currency} {magnitude}")
    if accounting and negative:
        return f"({base})"
    if negative:
        return f"-{base}" if symbol_style else f"{self.currency} -{magnitude}"
    return base

Money.__format__ = money_format

### Step 4 — check normal, accounting, and near-zero cases

In [14]:
assert format(usd) == "USD -1,234.57"
assert format(usd, "symbol") == "-$1,234.57"
assert format(usd, "accounting") == "(USD 1,234.57)"
assert format(usd, "symbol-accounting") == "($1,234.57)"
assert format(Money(Decimal("-0.004"), "USD"), "symbol") == "$0.00"
assert format(Money(Decimal("2.345"), "EUR"), "symbol") == "€2.34"
assert format(Money(Decimal("2.355"), "EUR"), "symbol") == "€2.36"
print("Money format tests passed")

Money format tests passed


### Step 5 — test the failure contract and explain one important limitation

A formatted monetary amount is **not** a safe input for arithmetic: it may contain grouping separators, a symbol, or parentheses. Always calculate with the original `Decimal` amounts. Real financial systems may also need currency-specific minor units and explicit transaction-level rounding policies.

In [15]:
expect_error(TypeError, lambda: Money(0.1, "USD"), "Decimal")
expect_error(ValueError, lambda: Money(Decimal("1"), "JPY"), "supported")
expect_error(ValueError, lambda: format(usd, ".2f"), "Money format")

basket = [Decimal("12.10"), Decimal("0.25"), Decimal("3.40")]
total = Money(sum(basket, Decimal("0")), "USD")
assert format(total, "symbol") == "$15.75"
print("Display and arithmetic remain separate:", format(total))

Display and arithmetic remain separate: USD 15.75


### Solution recap

The class stores exact, immutable numeric data; formatting chooses a carefully delimited display policy. Centralize rounding, document which styles are allowed, and handle negative zero explicitly.

---
# Problem 4 — human-readable byte sizes with rounding promotion

### Problem statement

Implement `DataSize` for a nonnegative integer byte count. Support `iec|N` (powers of 1024: B/KiB/MiB/...) and `si|N` (powers of 1000: B/kB/MB/...), with `N` between 0 and 3. The empty spec means `iec|1`.

A subtle bug occurs near unit boundaries. For example, `999_950` bytes is `999.95 kB`; at one decimal place this would render as `1000.0 kB`. Your algorithm should instead **promote the rounded value** to `1.0 MB`.

**Constraints:** never use a float for the byte count or conversion; reject negative and Boolean counts.

### Step 1 — define the data model and legal units

In [16]:
IEC_UNITS = ("B", "KiB", "MiB", "GiB", "TiB", "PiB")
SI_UNITS = ("B", "kB", "MB", "GB", "TB", "PB")

@dataclass(frozen=True)
class DataSize:
    bytes: int

    def __post_init__(self):
        if isinstance(self.bytes, bool) or not isinstance(self.bytes, int):
            raise TypeError("bytes must be an integer, not bool or float")
        if self.bytes < 0:
            raise ValueError("byte count must be nonnegative")

assert DataSize(1024).bytes == 1024
expect_error(ValueError, lambda: DataSize(-1))
expect_error(TypeError, lambda: DataSize(True))

'bytes must be an integer, not bool or float'

### Step 2 — parse first; compute only after validation

Separating parsing from conversion makes errors predictable. This parser intentionally rejects alternative spellings such as `IEC|1`, `si|04`, or an unbounded precision.

In [17]:
def parse_size_spec(spec):
    if spec == "":
        return "iec", 1
    parts = spec.split("|")
    if len(parts) != 2 or parts[0] not in {"iec", "si"}:
        raise ValueError("size spec must be iec|N or si|N")
    if parts[1] not in {"0", "1", "2", "3"}:
        raise ValueError("size precision N must be 0..3")
    return parts[0], int(parts[1])

assert parse_size_spec("") == ("iec", 1)
assert parse_size_spec("si|2") == ("si", 2)
expect_error(ValueError, lambda: parse_size_spec("si|04"))

'size precision N must be 0..3'

### Step 3 — choose units, then check for promotion *after rounding*

Find the largest unit whose unrounded size is at least one. Round to the requested precision using `Decimal.__format__`; if the resulting displayed number reaches the unit base, move to the next unit and compute again from the **original bytes**. Recomputing avoids cascading rounding errors.

In [18]:
def size_format(self, format_spec):
    system, precision = parse_size_spec(format_spec)
    base, units = (1024, IEC_UNITS) if system == "iec" else (1000, SI_UNITS)
    index = 0
    while index < len(units) - 1 and self.bytes >= base ** (index + 1):
        index += 1

    while True:
        scaled = Decimal(self.bytes) / Decimal(base ** index)
        text = format(scaled, f".{precision}f")
        if Decimal(text) >= base and index < len(units) - 1:
            index += 1
            continue
        return f"{text} {units[index]}"

DataSize.__format__ = size_format

### Step 4 — boundary tests (the main point of the problem)

Checking only friendly values like `1024` would miss the original bug. Test immediately below a unit boundary and at values where rounding changes the displayed unit.

In [19]:
assert format(DataSize(0)) == "0.0 B"
assert format(DataSize(42), "si|0") == "42 B"
assert format(DataSize(1024)) == "1.0 KiB"
assert format(DataSize(1000), "si|2") == "1.00 kB"
assert format(DataSize(999_950), "si|1") == "1.0 MB"
assert format(DataSize(1024**2 - 1), "iec|1") == "1.0 MiB"
assert format(DataSize(1536), "iec|2") == "1.50 KiB"
print("All size boundary cases passed")

All size boundary cases passed


### Step 5 — dynamic specs: the outer f-string builds the spec

A replacement field inside an f-string's format specification is resolved *before* calling `DataSize.__format__`. This is useful for programmatic precision, but the custom parser still validates the resulting specification.

In [20]:
precision = 2
size = DataSize(1536)
assert f"{size:iec|{precision}}" == "1.50 KiB"
for bad_spec in ("IEC|1", "si|4", "si", "si|1|extra"):
    expect_error(ValueError, lambda bad_spec=bad_spec: format(size, bad_spec))
print("Dynamic format spec:", f"{size:iec|{precision}}")

Dynamic format spec: 1.50 KiB


### Solution recap

A robust human-readable formatter needs a second check **after rounding**. Integer bytes and decimal scaling prevent float-boundary surprises; strict parsing prevents unsupported options from silently doing the wrong thing.

---
# Problem 5 — negative durations and three representations

### Problem statement

Wrap a `datetime.timedelta` and support:

- empty or `clock`: signed total hours as `HH:MM:SS` (hours may exceed 23);
- `days`: a signed `Nd HH:MM:SS` decomposition;
- `iso`: an ISO-8601-inspired duration such as `P1DT3H4M5S`.

The last form is intentionally restricted to whole seconds with all components present; the leading minus sign is outside the `P`. Require whole-second input and preserve negative durations correctly, even though `timedelta(days=-1, seconds=...)` stores negatives in normalized fields.

**Pitfall to avoid:** using `int(td.total_seconds())` for exact arithmetic or reading `td.seconds` as if it were a signed total.

### Step 1 — derive signed microseconds using integer arithmetic

`timedelta` stores days, seconds, and microseconds as integers. Combining those directly avoids float conversion and gives an exact total even for large durations.

In [21]:
from datetime import timedelta

MICROSECONDS_PER_SECOND = 1_000_000
MICROSECONDS_PER_DAY = 86_400 * MICROSECONDS_PER_SECOND

@dataclass(frozen=True)
class Duration:
    value: timedelta

    def __post_init__(self):
        if not isinstance(self.value, timedelta):
            raise TypeError("Duration requires datetime.timedelta")
        if self.total_microseconds % MICROSECONDS_PER_SECOND:
            raise ValueError("Duration in this exercise requires whole seconds")

    @property
    def total_microseconds(self):
        td = self.value
        return (td.days * MICROSECONDS_PER_DAY
                + td.seconds * MICROSECONDS_PER_SECOND
                + td.microseconds)

assert Duration(timedelta(seconds=-65)).total_microseconds == -65_000_000
expect_error(ValueError, lambda: Duration(timedelta(microseconds=1)))

'Duration in this exercise requires whole seconds'

### Step 2 — decompose the *absolute* duration

First peel off a single sign. Then `divmod` lets us recover days, hours, minutes, and seconds without negative-remainder surprises. The same decomposition works for all three output modes.

In [22]:
def duration_components(duration):
    total_seconds = duration.total_microseconds // MICROSECONDS_PER_SECOND
    sign = "-" if total_seconds < 0 else ""
    absolute = abs(total_seconds)
    days, remainder = divmod(absolute, 86_400)
    hours, remainder = divmod(remainder, 3600)
    minutes, seconds = divmod(remainder, 60)
    return sign, days, hours, minutes, seconds

assert duration_components(Duration(timedelta(seconds=-65))) == (
    "-", 0, 0, 1, 5
)

### Step 3 — format each documented representation

The `clock` form converts days back into *total* hours; the `days` form leaves hours in `00..23`. The ISO-style form preserves an explicit day count, including zero days, to keep the output structure uniform.

In [23]:
def duration_format(self, format_spec):
    mode = format_spec or "clock"
    if mode not in {"clock", "days", "iso"}:
        raise ValueError("Duration spec must be clock, days, or iso")
    sign, days, hours, minutes, seconds = duration_components(self)
    if mode == "clock":
        return f"{sign}{days * 24 + hours:02d}:{minutes:02d}:{seconds:02d}"
    if mode == "days":
        return f"{sign}{days}d {hours:02d}:{minutes:02d}:{seconds:02d}"
    return f"{sign}P{days}DT{hours}H{minutes}M{seconds}S"

Duration.__format__ = duration_format

### Step 4 — prove that negative values and long hours work

In [24]:
long_duration = Duration(timedelta(hours=27, minutes=4, seconds=5))
assert format(long_duration) == "27:04:05"
assert format(long_duration, "days") == "1d 03:04:05"
assert format(long_duration, "iso") == "P1DT3H4M5S"

negative_duration = Duration(timedelta(seconds=-65))
assert format(negative_duration) == "-00:01:05"
assert format(negative_duration, "days") == "-0d 00:01:05"
assert format(negative_duration, "iso") == "-P0DT0H1M5S"
assert format(Duration(timedelta()), "clock") == "00:00:00"
expect_error(ValueError, lambda: format(long_duration, "%H:%M"))
print("Duration outputs:", format(long_duration), format(long_duration, "iso"))

Duration outputs: 27:04:05 P1DT3H4M5S


### Solution recap

Normalize signs once, decompose with exact integer arithmetic, and choose the presentation only at the end. For fractional-second support, extend the grammar explicitly instead of silently dropping microseconds.

---
# Problem 6 — a network-address formatter with a secure, bounded grammar

### Problem statement

Create `IPv4Display` using the standard `ipaddress` module. Support:

| Spec | Output for `192.0.2.9` |
|---|---|
| empty or `dotted` | `192.0.2.9` |
| `hex` | `C0000209` |
| `binary` | four 8-bit groups separated by `.` |
| `network/N` | the containing CIDR network for prefix length `N` |

Accept only prefix lengths from 0 through 32. Why should we validate `N` ourselves rather than pass arbitrary input along to formatting expressions?

### Step 1 — rely on `ipaddress` for network correctness

The domain-specific type already knows how to represent an IPv4 address and calculate containing networks. Our class adds display modes, not IP arithmetic.

In [25]:
from ipaddress import IPv4Address, IPv4Network

@dataclass(frozen=True)
class IPv4Display:
    address: IPv4Address

    def __post_init__(self):
        if not isinstance(self.address, IPv4Address):
            raise TypeError("address must be an IPv4Address")

ip = IPv4Display(IPv4Address("192.0.2.9"))
assert str(ip.address) == "192.0.2.9"

### Step 2 — implement modes without interpreting untrusted format text as Python code

Each mode maps to an explicitly selected operation. For `network/N`, permit only a canonical decimal integer, then enforce bounds. Neither `eval` nor user-built attribute traversal is needed.

In [26]:
def ipv4_format(self, format_spec):
    spec = format_spec or "dotted"
    if spec == "dotted":
        return str(self.address)
    if spec == "hex":
        return format(int(self.address), "08X")
    if spec == "binary":
        return ".".join(format(int(octet), "08b")
                        for octet in str(self.address).split("."))
    if spec.startswith("network/"):
        prefix_text = spec.removeprefix("network/")
        if prefix_text not in {str(i) for i in range(33)}:
            raise ValueError("network prefix must be a decimal integer from 0 to 32")
        return str(IPv4Network((self.address, int(prefix_text)), strict=False))
    raise ValueError(f"Unsupported IPv4 format: {format_spec!r}")

IPv4Display.__format__ = ipv4_format

### Step 3 — test outputs and malformed specifications

In [27]:
assert format(ip) == "192.0.2.9"
assert format(ip, "hex") == "C0000209"
assert format(ip, "binary") == "11000000.00000000.00000010.00001001"
assert format(ip, "network/24") == "192.0.2.0/24"
assert format(ip, "network/0") == "0.0.0.0/0"
for invalid in ("network/33", "network/-1", "network/024", "network/x", "HEX"):
    expect_error(ValueError, lambda invalid=invalid: format(ip, invalid))
expect_error(TypeError, lambda: IPv4Display("192.0.2.9"))
print("Network formatter tests passed")

Network formatter tests passed


### Solution recap

Delegate domain work to dedicated standard-library types. An allowlisted grammar makes the accepted operations clear and prevents accidental implementation complexity. This example does not do DNS lookup or external network access.

---
# Problem 7 — combine coordinate systems with Python's numeric mini-language

### Problem statement

Implement a finite 2D vector with two coordinate modes:

- empty or `xy`: `x=3.00, y=4.00`
- `polar`: `r=5.00, θ=53.13°`

Allow an optional numeric spec after a pipe: `xy|+08.2f`, `polar|.3f`, etc. Your custom grammar chooses the *coordinate system* while standard `float.__format__` handles numeric presentation.

**Design challenge:** Avoid writing a fragile parser for numeric widths, signs, precision, alignment, or grouping; Python already has one. Reject NaN and infinities because they make geometric reporting ambiguous.

### Step 1 — choose immutable, finite input data

In [28]:
from math import atan2, degrees, hypot, isfinite

@dataclass(frozen=True)
class Vector2D:
    x: float
    y: float

    def __post_init__(self):
        if not all(isinstance(v, (int, float)) and not isinstance(v, bool)
                   and isfinite(v) for v in (self.x, self.y)):
            raise ValueError("both vector coordinates must be finite real numbers")

v = Vector2D(3.0, 4.0)
assert hypot(v.x, v.y) == 5.0
expect_error(ValueError, lambda: Vector2D(float("nan"), 1.0))

'both vector coordinates must be finite real numbers'

### Step 2 — parse the domain prefix and retain the numeric suffix intact

`partition('|')` splits only at the first separator. We do not try to inspect `.3f` or `+08.2f`: these are passed unchanged to `format(float_value, numeric_spec)`, which validates them according to Python's documented numeric mini-language.

In [29]:
def parse_vector_spec(spec):
    if not spec:
        return "xy", ".2f"
    mode, separator, numeric_spec = spec.partition("|")
    if mode not in {"xy", "polar"}:
        raise ValueError("vector mode must be xy or polar")
    if separator and not numeric_spec:
        raise ValueError("numeric spec after '|' cannot be empty")
    return mode, numeric_spec if separator else ".2f"

assert parse_vector_spec("polar|.3f") == ("polar", ".3f")
assert parse_vector_spec("xy") == ("xy", ".2f")
expect_error(ValueError, lambda: parse_vector_spec("polar|"))

"numeric spec after '|' cannot be empty"

### Step 3 — implement the geometry and delegate presentation

Angles are measured counterclockwise from the positive x-axis, in degrees. The string contains the **unit marker** `°` so a reader cannot confuse degrees and radians.

In [30]:
def vector_format(self, format_spec):
    mode, number_spec = parse_vector_spec(format_spec)
    if mode == "xy":
        x_text = format(self.x, number_spec)
        y_text = format(self.y, number_spec)
        return f"x={x_text}, y={y_text}"
    radius = hypot(self.x, self.y)
    angle_degrees = degrees(atan2(self.y, self.x))
    return (f"r={format(radius, number_spec)}, "
            f"θ={format(angle_degrees, number_spec)}°")

Vector2D.__format__ = vector_format

### Step 4 — exercise defaults, nested specs, and delegated errors

`f'{v:xy|{numeric_spec}}'` uses nested f-string substitution to **construct** the custom spec before your `__format__` receives it. A malformed numeric mini-language should raise an error rather than silently returning an arbitrary string.

In [31]:
assert format(v) == "x=3.00, y=4.00"
assert format(v, "polar") == "r=5.00, θ=53.13°"
assert format(v, "xy|+08.2f") == "x=+0003.00, y=+0004.00"
numeric_spec = ".3f"
assert f"{v:polar|{numeric_spec}}" == "r=5.000, θ=53.130°"
expect_error(ValueError, lambda: format(v, "spherical|.2f"))
expect_error(ValueError, lambda: format(v, "xy|.2q"))
print(format(v, "polar|.3f"))

r=5.000, θ=53.130°


### Solution recap

Custom specs need only express behavior that belongs to the domain; the underlying numeric object handles the rest. Be explicit about coordinate systems, angle units, and invalid numeric values.

---
# Problem 8 — timezone-aware timestamp formatting without ambiguous local time

### Problem statement

Create a `Timestamp` wrapper that **rejects naive datetimes**. The empty spec prints an ISO-8601 timestamp in UTC at second precision. Two additional spec families should delegate to `datetime.__format__`:

- `utc|%Y-%m-%d %H:%M %z` — convert to UTC first;
- `source|%Y-%m-%d %H:%M %z` — preserve the supplied timezone offset.

Use a fixed datetime in UTC+02:00 to show that the same instant has two valid clock representations. Do not call `datetime.now()` or depend on the machine's local timezone.

### Step 1 — enforce awareness at the boundary

A datetime is aware when its `tzinfo` is present **and** `utcoffset()` is not `None`. The second condition matters because custom timezone implementations may return `None`.

In [32]:
from datetime import datetime, timezone

@dataclass(frozen=True)
class Timestamp:
    value: datetime

    def __post_init__(self):
        if not isinstance(self.value, datetime):
            raise TypeError("Timestamp requires datetime")
        if self.value.tzinfo is None or self.value.utcoffset() is None:
            raise ValueError("Timestamp requires a timezone-aware datetime")

fixed_source = datetime(2025, 5, 6, 12, 34, 56,
                        tzinfo=timezone(timedelta(hours=2)))
ts = Timestamp(fixed_source)
expect_error(ValueError, lambda: Timestamp(datetime(2025, 5, 6, 12, 34)))

'Timestamp requires a timezone-aware datetime'

### Step 2 — distinguish a stable default from custom `strftime` patterns

The default is UTC ISO text with an explicit `+00:00` offset. When users opt into `utc|...` or `source|...`, everything after the first `|` is passed as the datetime's own format specification. It may contain spaces and multiple percent directives, so **do not split it again**.

In [33]:
def timestamp_format(self, format_spec):
    utc_value = self.value.astimezone(timezone.utc)
    if format_spec == "":
        return utc_value.isoformat(timespec="seconds")

    mode, separator, pattern = format_spec.partition("|")
    if not separator or mode not in {"utc", "source"} or not pattern:
        raise ValueError("timestamp spec must be utc|PATTERN or source|PATTERN")
    target = utc_value if mode == "utc" else self.value
    return format(target, pattern)  # Delegate to datetime.__format__.

Timestamp.__format__ = timestamp_format

### Step 3 — verify both display zones refer to the same instant

The offsets in the expected output make the change visible. The wrapper never implicitly consults a system time zone, so tests are stable on different computers.

In [34]:
assert format(ts) == "2025-05-06T10:34:56+00:00"
assert format(ts, "utc|%Y-%m-%d %H:%M %z") == "2025-05-06 10:34 +0000"
assert format(ts, "source|%Y-%m-%d %H:%M %z") == "2025-05-06 12:34 +0200"
assert format(ts, "utc|%Y/%m/%d") == "2025/05/06"
for invalid in ("utc", "local|%H:%M", "source|", "%Y-%m-%d"):
    expect_error(ValueError, lambda invalid=invalid: format(ts, invalid))
print("Timestamp is locale- and machine-timezone-independent for the tested directives")

Timestamp is locale- and machine-timezone-independent for the tested directives


### Solution recap

`datetime.__format__` already interprets `strftime` directives, so reuse it. Distinguish source offset from UTC explicitly, validate timezone awareness, and use numeric directives when deterministic output matters. Arbitrary `strftime` directives can be locale/platform dependent; the tested patterns deliberately avoid those variations.

---
# Problem 9 — compose objects into an invoice line and safe CSV

### Problem statement

Use the existing `Money` implementation inside a new `InvoiceLine` class. Implement:

- empty or `row|N`: a fixed-width row whose SKU column has width `N` (default 12);
- `receipt`: a human-readable description with symbol-formatted prices;
- `csv`: a **single, properly escaped CSV record** with SKU, quantity, unit price, and total.

Validate quantity, SKU, width, and the supported grammar. A SKU may contain commas and quotes, so joining fields with `','.join(...)` is not correct CSV. In this exercise, row widths are 4 through 30 and too-long SKUs trigger an error rather than silent truncation.

### Step 1 — separate business data from derived totals

Compute the line total from the original `Decimal` amount. Do not parse the output of `Money.__format__`: that output exists for humans, not for arithmetic.

In [35]:
import csv
from io import StringIO

@dataclass(frozen=True)
class InvoiceLine:
    sku: str
    quantity: int
    unit_price: Money

    def __post_init__(self):
        if not isinstance(self.sku, str) or not self.sku:
            raise ValueError("sku must be nonempty text")
        if isinstance(self.quantity, bool) or not isinstance(self.quantity, int) \
                or self.quantity <= 0:
            raise ValueError("quantity must be a positive integer")
        if not isinstance(self.unit_price, Money):
            raise TypeError("unit_price must be Money")

    @property
    def total(self):
        return Money(self.unit_price.amount * self.quantity,
                     self.unit_price.currency)

line = InvoiceLine("USB-C cable", 3, Money(Decimal("8.50"), "USD"))
assert line.total.amount == Decimal("25.50")

### Step 2 — parse width only for table rows

The mode `row|N` is a tiny custom grammar. Restrict `N` before placing it inside a standard string width spec. `receipt` and `csv` have no extra options, keeping their meanings unambiguous.

In [36]:
def parse_invoice_spec(spec):
    if spec == "":
        return "row", 12
    if spec in {"receipt", "csv"}:
        return spec, None
    if spec.startswith("row|"):
        width_text = spec.removeprefix("row|")
        if width_text not in {str(n) for n in range(4, 31)}:
            raise ValueError("row width must be an integer from 4 to 30")
        return "row", int(width_text)
    raise ValueError("invoice spec must be row|N, receipt, or csv")

assert parse_invoice_spec("") == ("row", 12)
assert parse_invoice_spec("row|16") == ("row", 16)
expect_error(ValueError, lambda: parse_invoice_spec("row|-2"))

'row width must be an integer from 4 to 30'

### Step 3 — delegate formatting to `Money` and CSV escaping to `csv.writer`

For a row, use standard string/number alignment *after* formatting the money fields. For CSV, make `csv.writer` responsible for commas, quotes, and newlines; pass the original fields as values, not a preassembled comma-separated string.

In [37]:
def invoice_format(self, format_spec):
    mode, width = parse_invoice_spec(format_spec)
    if mode == "receipt":
        return (f"{self.quantity} × {self.sku} @ "
                f"{format(self.unit_price, 'symbol')} = "
                f"{format(self.total, 'symbol')}")
    if mode == "csv":
        buffer = StringIO(newline="")
        writer = csv.writer(buffer, lineterminator="\n")
        writer.writerow([self.sku, self.quantity,
                         format(self.unit_price), format(self.total)])
        return buffer.getvalue().removesuffix("\n")

    if len(self.sku) > width or "\n" in self.sku or "\r" in self.sku:
        raise ValueError("SKU does not fit safely in the requested row width")
    sku_text = format(self.sku, f"<{width}")
    quantity_text = format(self.quantity, ">4d")
    unit_text = format(format(self.unit_price), ">18")
    total_text = format(format(self.total), ">18")
    return f"{sku_text} {quantity_text} {unit_text} {total_text}"

InvoiceLine.__format__ = invoice_format

### Step 4 — verify alignment and dynamic widths

Comparing with a known expression is clearer than checking whitespace by eye. A nested f-string field can construct the `row|N` spec dynamically, provided our parser validates the resulting width.

In [38]:
assert format(line, "receipt") == "3 × USB-C cable @ $8.50 = $25.50"
assert format(line) == (
    f"{'USB-C cable':<12} {3:>4d} {'USD 8.50':>18} {'USD 25.50':>18}"
)
width = 16
assert f"{line:row|{width}}" == format(line, "row|16")
expect_error(ValueError, lambda: format(line, "row|3"))
expect_error(ValueError, lambda: format(line, "row|31"))
print(format(line, "row|16"))

USB-C cable         3           USD 8.50          USD 25.50


### Step 5 — prove that CSV survives troublesome SKU characters

The meaningful CSV test is a **round-trip through a CSV parser**, not a visual comparison. This catches failures when a SKU contains embedded quotes, commas, or newline characters.

In [39]:
tricky = InvoiceLine('Widget, "large"\nblue', 2,
                     Money(Decimal("19.95"), "EUR"))
csv_text = format(tricky, "csv")
parsed = next(csv.reader(StringIO(csv_text, newline="")))
assert parsed == [tricky.sku, "2", "EUR 19.95", "EUR 39.90"]
expect_error(ValueError, lambda: format(tricky, "row|30"))
expect_error(ValueError, lambda: InvoiceLine("", 1, line.unit_price))
expect_error(ValueError, lambda: InvoiceLine("OK", 0, line.unit_price))
print("Quoted CSV record round-tripped safely")

Quoted CSV record round-tripped safely


### Solution recap

Objects with good `__format__` methods can be composed, but formatting is still only a **presentation** API. Use original numeric values for calculations, reject truncation in fixed-width views, and use a real serializer for machine-readable formats.

---
# Problem 10 — capstone: a telemetry record with table and JSON views

### Problem statement

Build a `TelemetryRecord` combining four independent kinds of data: an aware `Timestamp`, a source name, a `DataSize`, and a `Vector2D`. Implement:

- empty or `table|N`: a readable, aligned table row with source-column width `N` (default 12, permitted 8..24);
- `json`: machine-readable JSON with the **original byte count and coordinates**, plus a UTC ISO timestamp.

Write a report with a header and two rows. Include a source containing quotation marks and a comma to prove that JSON correctly escapes it. Reject overlong source names in table mode and unknown specs.

**Architecture question:** Which component should handle UTC conversion, unit rounding, vector-coordinate formatting, JSON string escaping, and column alignment? Give each responsibility to the component or library that already owns it.

### Step 1 — define a validated composite data model

The record does not convert bytes or coordinates into strings during construction. Retaining typed objects lets each output view make its own choices without information loss.

In [40]:
import json

@dataclass(frozen=True)
class TelemetryRecord:
    timestamp: Timestamp
    source: str
    payload: DataSize
    displacement: Vector2D

    def __post_init__(self):
        if not isinstance(self.timestamp, Timestamp):
            raise TypeError("timestamp must be Timestamp")
        if not isinstance(self.source, str) or not self.source:
            raise ValueError("source must be nonempty text")
        if not isinstance(self.payload, DataSize):
            raise TypeError("payload must be DataSize")
        if not isinstance(self.displacement, Vector2D):
            raise TypeError("displacement must be Vector2D")

record = TelemetryRecord(ts, "sensor-A", DataSize(1536), Vector2D(3, 4))
assert record.payload.bytes == 1536

### Step 2 — set a simple top-level grammar

The record owns only the view selection and table column width. It delegates UTC formatting to `Timestamp`, data-size rounding to `DataSize`, polar presentation to `Vector2D`, and machine-readable escaping to `json.dumps`.

In [41]:
def parse_telemetry_spec(spec):
    if spec == "":
        return "table", 12
    if spec == "json":
        return "json", None
    if spec.startswith("table|"):
        width_text = spec.removeprefix("table|")
        if width_text not in {str(n) for n in range(8, 25)}:
            raise ValueError("table source width must be 8..24")
        return "table", int(width_text)
    raise ValueError("telemetry spec must be table|N or json")

assert parse_telemetry_spec("table|14") == ("table", 14)
expect_error(ValueError, lambda: parse_telemetry_spec("table|1000"))

'table source width must be 8..24'

### Step 3 — implement a human row and a lossless structured view

In table mode, choose display-oriented specs for the nested objects. JSON mode instead extracts typed properties directly. In particular, the exact integer `bytes` value is **not** reconstructed from a rounded `KiB` string. `json.dumps` handles quotation marks and control characters. The example specifies sorted keys for deterministic tests.

In [42]:
def telemetry_format(self, format_spec):
    mode, width = parse_telemetry_spec(format_spec)
    if mode == "json":
        payload = {
            "timestamp_utc": self.timestamp.value.astimezone(
                timezone.utc).isoformat(timespec="seconds"),
            "source": self.source,
            "bytes": self.payload.bytes,
            "x": self.displacement.x,
            "y": self.displacement.y,
        }
        return json.dumps(payload, sort_keys=True, allow_nan=False)

    if len(self.source) > width or any(c in self.source for c in "\r\n"):
        raise ValueError("source does not fit safely in the table width")
    timestamp_text = format(self.timestamp, "utc|%Y-%m-%d %H:%M")
    source_text = format(self.source, f"<{width}")
    size_text = format(format(self.payload, "iec|1"), ">12")
    displacement_text = format(self.displacement, "polar|.1f")
    return f"{timestamp_text}  {source_text}  {size_text}  {displacement_text}"

TelemetryRecord.__format__ = telemetry_format

### Step 4 — assert correctness before printing the report

Test both modes independently. For JSON, decode the string and inspect the actual values. For the table, verify the fixed-width prefix and expected delegated representations.

In [43]:
row = format(record, "table|14")
assert row.startswith("2025-05-06 10:34  sensor-A      ")
assert "1.5 KiB" in row
assert row.endswith("r=5.0, θ=53.1°")

structured = json.loads(format(record, "json"))
assert structured == {
    "timestamp_utc": "2025-05-06T10:34:56+00:00",
    "source": "sensor-A", "bytes": 1536, "x": 3, "y": 4,
}
print("Both telemetry views passed independent tests")

Both telemetry views passed independent tests


### Step 5 — build a real, repeatable two-row report

Construct the header using the *same source-column width* as the rows. Keep a single separator convention rather than aligning output manually. The `table|N` spec is still validated by every record.

In [44]:
record_2 = TelemetryRecord(
    Timestamp(datetime(2025, 5, 6, 9, 0, tzinfo=timezone.utc)),
    "sensor-B", DataSize(1024**2), Vector2D(0, 2)
)

source_width = 14
header = (f"{'UTC timestamp':16}  {'Source':<{source_width}}  "
          f"{'Payload':>12}  {'Displacement (polar)'}")
separator = "-" * len(header)
report = "\n".join([
    header, separator,
    *(format(item, f"table|{source_width}") for item in (record, record_2)),
])
assert report.count("\n") == 3
assert "sensor-B" in report and "1.0 MiB" in report
print(report)

UTC timestamp     Source               Payload  Displacement (polar)
--------------------------------------------------------------------
2025-05-06 10:34  sensor-A             1.5 KiB  r=5.0, θ=53.1°
2025-05-06 09:00  sensor-B             1.0 MiB  r=2.0, θ=90.0°


### Step 6 — verify escaping, validation, and nontruncation

A `source` string with a comma and quotation marks needs **no hand-written escaping** in JSON. Conversely, a long source in a fixed-width table should fail loudly, not silently change the identity being reported.

In [45]:
quoted = TelemetryRecord(ts, 'sensor, "prototype"', DataSize(8), Vector2D(0, 0))
assert json.loads(format(quoted, "json"))["source"] == 'sensor, "prototype"'
expect_error(ValueError, lambda: format(quoted, "table|8"), "fit")
for invalid in ("table", "table|7", "table|25", "yaml", "json|pretty"):
    expect_error(ValueError, lambda invalid=invalid: format(record, invalid))
print("Escaping and invalid-spec tests passed")

Escaping and invalid-spec tests passed


### Capstone solution recap

The outer class orchestrates **what** view to produce; component classes and serialization libraries determine **how** individual fields are represented. A human-readable table can round units for readability; a machine-readable JSON representation preserves raw values. Those are different contracts and should not be mixed.

---
# Final checklist — best practices for production `__format__`

Before publishing a formattable class, verify the following:

1. **Empty spec:** Decide and test what `format(obj)` and `f'{obj}'` mean.
2. **Result type:** Every successful path returns `str`, never `None`, bytes, or a number.
3. **Clear grammar:** Document the complete set of custom specs and reject unknown options.
4. **Delegation:** Use `format(self.amount, '.2f')`, `format(self.date, '%Y')`, and other built-in formatters instead of rewriting their mini-languages.
5. **No recursion:** Never render `self` through a plain f-string inside its own `__format__`.
6. **Separate model and view:** Formatting must not mutate stored data or become the source for arithmetic.
7. **Precision and signs:** Check rounding, boundaries, negative zero, and long/negative durations.
8. **Deterministic tests:** Use fixed instants and known locale-independent patterns.
9. **Safe output:** Use `csv.writer` or `json.dumps` rather than handcrafted serializers; validate custom widths and modes.
10. **Coverage:** Test each legal spec, empty spec, malformed specs, and relevant edge cases.

## Additional independent practice (solutions are intentionally left to you)

**A. Scientific measurements:** Extend `Vector2D` with a `norm|NUMERIC_SPEC` mode and check zero vectors, precision, and signs. Keep the existing output unchanged.

**B. Non-ASCII and display width:** Revisit `InvoiceLine` when SKU names contain combining characters or East Asian wide glyphs. Explain why `len(text)` and Python's alignment widths do not guarantee terminal-cell alignment. Propose a display library for real terminal UIs rather than claiming this notebook solves Unicode cell widths.

**C. Financial policy:** Refactor `Money` so each currency has a documented minor-unit rule, and distinguish rounding per item from rounding at the total. State assumptions and test both policies with exact decimal values.

**D. Duration subsecond support:** Extend `Duration` with an explicit `clock|N` grammar for milliseconds/microseconds. Preserve the exact stored microseconds and define how negative fractional durations are displayed.

### End-to-end smoke test

The final cell exercises one representative path through every problem. If it runs without an assertion failure, the notebook's ten main solutions are available in the current kernel.

In [46]:
assert format(ProtocolProbe("Ada"), ">5") == "  Ada"
assert format(FractionView(1, 8), "pct|1") == "12.5%"
assert format(Money(Decimal("1.00"), "USD"), "symbol") == "$1.00"
assert format(DataSize(999_950), "si|1") == "1.0 MB"
assert format(Duration(timedelta(hours=27)), "days") == "1d 03:00:00"
assert format(IPv4Display(IPv4Address("192.0.2.9")), "network/24") == "192.0.2.0/24"
assert format(Vector2D(3.0, 4.0), "polar") == "r=5.00, θ=53.13°"
assert format(ts).endswith("+00:00")
assert next(csv.reader(StringIO(format(line, "csv"))))[0] == "USB-C cable"
assert json.loads(format(record, "json"))["bytes"] == 1536
print("SUCCESS: all 10 problem solutions passed the end-to-end smoke test.")

SUCCESS: all 10 problem solutions passed the end-to-end smoke test.
